# Finance ML Analytics Platform — v8_3

**Version 0.3.0** — Now using modular `finance_ml` package

## What's New

- All functions are now imported from the `finance_ml` package
- No need to define functions inline — they're maintained in the package modules
- Configuration management with `FinanceMLConfig`
- Better code organization and testability
- Feature flags for optional functionality control

## Modules

- `finance_ml.data`: Data loading, normalization, validation
- `finance_ml.features`: Feature engineering
- `finance_ml.models`: Classification, regression, ensembles
- `finance_ml.eval`: Analytics, visualizations, reporting
- `finance_ml.config`: Configuration management
- `finance_ml.cli`: Command-line interface

## Usage

This notebook demonstrates the ML workflow:
1. Load and validate data
2. Exploratory data analysis
3. Feature engineering
4. Model training (classification and regression)
5. Evaluation and analytics

## Configuration and Feature Flags

In [ ]:
# Feature Availability via NotebookConfig
# Centralize feature flags using finance_ml.NotebookConfig; keep legacy variables for compatibility
from finance_ml import NotebookConfig

cfg = NotebookConfig(
        have_finance_prediction=True,
        have_database_connection=False,
        have_advanced_analytics=True,
        have_dim_reduction=False,
        debug_mode=False,
        enable_sector_analysis=True,
        enable_region_analysis=True,
        enable_interactive_plots=True,
        enable_excel_export=True,
        )

# Display a concise summary using the tested display API
cfg.display_summary()

# Backward-compatible variables (used later in the notebook)
HAVE_FINANCE_PREDICTION = cfg.have_finance_prediction
HAVE_DATABASE_CONNECTION = cfg.have_database_connection
HAVE_ADVANCED_ANALYTICS = cfg.have_advanced_analytics
HAVE_DIM_REDUCTION = cfg.have_dim_reduction
DEBUG_MODE = cfg.debug_mode
ENABLE_SECTOR_ANALYSIS = cfg.enable_sector_analysis
ENABLE_REGION_ANALYSIS = cfg.enable_region_analysis
ENABLE_INTERACTIVE_PLOTS = cfg.enable_interactive_plots
ENABLE_EXCEL_EXPORT = cfg.enable_excel_export

In [ ]:
# Finance ML Analytics Platform — Notebook (v0.3.0)
# This notebook now uses the modular finance_ml package

import warnings

warnings.filterwarnings('ignore')

# Data science libraries
import numpy as np
import pandas as pd

# Import all functions from finance_ml package
from finance_ml import (
    # Version
    __version__,
    # Configuration
    load_config,
    # Utilities
    setup_logging,
    # Data loading and validation
    preprocess,
    # Notebook utilities
    display_config_summary,
    load_stock_data,
    display_data_summary,
    # Feature engineering
    build_features_and_target,
    # Modeling
    create_event_labels,
    train_event_classifier,
    train_and_evaluate_regression,
    # Evaluation and analytics
    calculate_mispricing_score,
    rank_undervalued_stocks,
    rank_overvalued_stocks,
    create_sector_heatmap,
    create_interactive_prediction_plot,
    export_predictions_to_excel,
    # Week 1 Enhancements - Data Quality & Monitoring
    validate_financial_data_quality,
    sanitize_dataframe_with_logging,
    monitor_ensemble_training,
    perform_early_pipeline_validation,
    )

# Setup logging
import logging

setup_logging()
logger = logging.getLogger(__name__)

print(f"Finance ML Analytics Platform v{__version__}")
print("All functions imported from finance_ml package")


## Configuration

Load configuration from environment variables or config files.


In [ ]:
# Load configuration
config = load_config()
display_config_summary(config)


## Sample Data Generator

Create sample financial dataset for demonstration when real data is unavailable.

Note: The generator is now provided by the package as
`finance_ml.create_sample_financial_dataset` — no inline definition needed here.


## Data Loading

Load stock data from configured data source (database or CSV files) with automatic fallback to sample data.


In [ ]:
# Load stock data using package strategy helpers
all_stocks = load_stock_data(config)
if all_stocks is None or len(all_stocks) == 0:
    raise ValueError("Failed to load any stock data")

display_data_summary(all_stocks)


## Data Validation and Quality Checks

Validate schema and check data quality using finance_ml package functions.


In [ ]:
# Unified validation reporting with error handling
try:
    from finance_ml.data import validate_schema, check_missing_values

    # Schema validation
    try:
        validate_schema(all_stocks)
        print("✓ Schema validation passed")
    except Exception as e:
        print(f"⚠ Schema validation warning: {e}")

    # Missing values check
    try:
        missing_report = all_stocks.isnull().sum()
        missing_pct = (missing_report / len(all_stocks) * 100).round(2)
        missing_df = pd.DataFrame({
            'Missing Count': missing_report[missing_report > 0],
            'Missing %': missing_pct[missing_report > 0]
            }).sort_values('Missing Count', ascending=False)

        if len(missing_df) > 0:
            print("\n📊 Missing Values Report:")
            print(missing_df.head(10).to_string())
        else:
            print("✓ No missing values detected")
    except Exception as e:
        print(f"⚠ Missing value check failed: {e}")

except Exception as e:
    logger.error(f"Validation failed: {e}")
    print(f"⚠ Validation checks incomplete")

## Exploratory Data Analysis

Perform EDA using the simple_eda function.


In [ ]:
# Unified EDA display with proper output directory
try:
    from pathlib import Path

    output_dir = Path(config.output_dir)
    output_dir.mkdir(exist_ok=True, parents=True)

    from finance_ml.eval import simple_eda

    print("\n" + "=" * 80)
    print("EXPLORATORY DATA ANALYSIS")
    print("=" * 80)

    simple_eda(all_stocks, out_dir=output_dir)
    print(f"✓ EDA completed - outputs saved to {output_dir}")

except Exception as e:
    logger.error(f"EDA failed: {e}")
    print(f"⚠ EDA failed: {e}")

## Data Preprocessing

Preprocess and normalize the data using finance_ml package.


In [ ]:
# Preprocess data
try:
    all_stocks_processed = preprocess(all_stocks)
    logger.info(f"Data preprocessed: {all_stocks_processed.shape}")
    print(f"\n✓ Data preprocessed successfully")
    print(f"  Shape after preprocessing: {all_stocks_processed.shape}")
except Exception as e:
    logger.error(f"Preprocessing failed: {e}")
    print(f"✗ Preprocessing failed: {e}")
    all_stocks_processed = all_stocks.copy()


## Feature Engineering

Build features using the complete feature pipeline from finance_ml package.


In [ ]:
# Build features and target
try:
    X, y, numeric_features, categorical_features = build_features_and_target(all_stocks_processed)
    logger.info(f"Features built: {X.shape}, Target: {y.shape if y is not None else 'None'}")
    print(f"\n✓ Features engineered successfully")
    print(f"  Feature matrix shape: {X.shape}")
    print(f"  Target shape: {y.shape if y is not None else 'None'}")
    print(f"  Numeric features: {len(numeric_features)}")
    print(f"  Categorical features: {len(categorical_features)}")
    print(f"  First 10 numeric features: {numeric_features[:10]}")
    if categorical_features:
        print(f"  Categorical features: {categorical_features}")
except Exception as e:
    logger.error(f"Feature engineering failed: {e}")
    print(f"✗ Feature engineering failed: {e}")
    raise

## Model Training - Classification

Train event classifier using finance_ml package.


In [ ]:
# Create event labels for classification
try:
    event_labels = create_event_labels(all_stocks_processed)
    print(f"\n✓ Event labels created")
    print(f"  Label distribution: {pd.Series(event_labels).value_counts().to_dict()}")

    # Remove duplicate columns before training
    df_for_classifier = all_stocks_processed.copy()
    duplicate_cols = df_for_classifier.columns[df_for_classifier.columns.duplicated()].unique()

    if len(duplicate_cols) > 0:
        print(f"  Removing duplicate columns: {list(duplicate_cols)}")
        # Keep only first occurrence of each column
        df_for_classifier = df_for_classifier.loc[:, ~df_for_classifier.columns.duplicated()]

    # Train event classifier using the cleaned DataFrame
    classifier_results = train_event_classifier(df_for_classifier, event_labels)
    print(f"\n✓ Event classifier trained")
    print(f"  Accuracy: {classifier_results.get('accuracy', 0):.4f}")
    print(f"  F1 Score (macro): {classifier_results.get('f1_macro', 0):.4f}")

except Exception as e:
    logger.error(f"Classification training failed: {e}")
    print(f"✗ Classification training failed: {e}")

## Model Training - Regression

Train regression models using finance_ml package.


In [ ]:
# Train baseline regression model
try:
    # Use the proper training function with required parameters
    from pathlib import Path

    output_dir = Path(config.output_dir)
    output_dir.mkdir(exist_ok=True, parents=True)

    regression_results = train_and_evaluate_regression(
            all_stocks_processed,
            out_dir=output_dir,
            n_jobs=config.n_jobs if hasattr(config, 'n_jobs') else -1
            )

    if regression_results:
        print(f"\n✓ Regression model trained")
        print(f"  MAE: {regression_results.get('mae', 0):.4f}")
        print(f"  RMSE: {regression_results.get('rmse', 0):.4f}")
        print(f"  R²: {regression_results.get('r2', 0):.4f}")

        # Store predictions in dataframe for later use
        if 'predictions' in regression_results:
            pred_df = regression_results['predictions']
            all_stocks_processed.loc[pred_df.index, 'predicted_target'] = pred_df['y_pred'].values
    else:
        print("⚠ Regression training skipped (insufficient data or dry run)")
except Exception as e:
    logger.error(f"Regression training failed: {e}")
    print(f"✗ Regression training failed: {e}")

## Stock Valuation Analysis

Calculate mispricing scores and rank stocks using finance_ml package.


In [ ]:
# Calculate mispricing scores
try:
    # Get predictions from regression model
    if 'predictions' in regression_results and regression_results['predictions'] is not None:
        predictions = regression_results['predictions']

        # Properly align predictions with dataframe using index
        if isinstance(predictions, pd.DataFrame):
            # predictions is a DataFrame with y_pred column
            pred_series = predictions['y_pred']
            all_stocks_processed.loc[pred_series.index, 'predicted_target'] = pred_series.values
        elif isinstance(predictions, pd.Series):
            all_stocks_processed.loc[predictions.index, 'predicted_target'] = predictions.values
        else:
            print("⚠ Unexpected prediction format")
            raise ValueError("Predictions must be DataFrame or Series")

        # Calculate mispricing only for rows with predictions
        mask = all_stocks_processed['predicted_target'].notna()
        df_with_pred = all_stocks_processed[mask].copy()

        mispricing = calculate_mispricing_score(df_with_pred)
        all_stocks_processed.loc[mask, 'mispricing_score'] = mispricing

        print(f"\n✓ Mispricing scores calculated for {mask.sum()} stocks")
        print(f"  Mean mispricing: {mispricing.mean():.4f}")
        print(f"  Std mispricing: {mispricing.std():.4f}")

        # Rank undervalued stocks (only from rows with mispricing scores)
        df_scored = all_stocks_processed[all_stocks_processed['mispricing_score'].notna()].copy()

        if len(df_scored) >= 10:
            undervalued = rank_undervalued_stocks(df_scored, top_n=10)
            print(f"\nTop 10 Undervalued Stocks:")
            display_cols = [c for c in ['ticker', 'name', 'sector', 'mispricing_score'] if c in undervalued.columns]
            print(undervalued[display_cols].to_string())

            # Rank overvalued stocks
            overvalued = rank_overvalued_stocks(df_scored, top_n=10)
            print(f"\nTop 10 Overvalued Stocks:")
            print(overvalued[display_cols].to_string())
        else:
            print(f"⚠ Insufficient scored stocks ({len(df_scored)}) for ranking")
    else:
        print("⚠ No predictions available from regression model")

except Exception as e:
    logger.error(f"Valuation analysis failed: {e}")
    print(f"✗ Valuation analysis failed: {e}")
    import traceback

    traceback.print_exc()

## Advanced Preprocessing Pipeline Demo

Demonstrate the proper use of separate transformers for numeric vs categorical features using the returned feature lists.

In [ ]:
# Demonstrate proper preprocessing pipeline with separate transformers
try:
    from sklearn.compose import ColumnTransformer
    from sklearn.preprocessing import StandardScaler, OneHotEncoder
    from sklearn.pipeline import Pipeline
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.model_selection import train_test_split

    if y is not None and len(X) > 0:
        print("\n" + "=" * 80)
        print("ADVANCED PREPROCESSING PIPELINE DEMONSTRATION")
        print("=" * 80)

        # Show feature type separation
        print(f"\n📊 Feature Type Analysis:")
        print(f"  Total features: {len(numeric_features) + len(categorical_features)}")
        print(f"  Numeric features: {len(numeric_features)}")
        print(f"  Categorical features: {len(categorical_features)}")

        # Build preprocessing pipeline with separate transformers
        print(f"\n🔧 Building preprocessing pipeline with separate transformers...")

        preprocessor = ColumnTransformer(
                transformers=[
                    ('numeric', StandardScaler(with_mean=False), numeric_features),
                    ('categorical', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
                    ],
                remainder='drop'
                )

        # Create full pipeline with regressor
        pipeline = Pipeline([
            ('preprocessor', preprocessor),
            ('regressor', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
            ])

        # Prepare data (remove NaN targets)
        mask = ~y.isna()
        X_clean = X.loc[mask]
        y_clean = y.loc[mask]

        if len(X_clean) >= 20:
            # Split data
            X_train, X_test, y_train, y_test = train_test_split(
                    X_clean, y_clean, test_size=0.2, random_state=42
                    )

            print(f"\n📈 Training pipeline...")
            print(f"  Training samples: {len(X_train)}")
            print(f"  Test samples: {len(X_test)}")

            # Fit pipeline
            pipeline.fit(X_train, y_train)

            # Make predictions
            y_pred = pipeline.predict(X_test)

            # Calculate metrics
            from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

            mae = mean_absolute_error(y_test, y_pred)
            rmse = np.sqrt(mean_squared_error(y_test, y_pred))
            r2 = r2_score(y_test, y_pred)

            print(f"\n✓ Pipeline training completed")
            print(f"  MAE: {mae:.4f}")
            print(f"  RMSE: {rmse:.4f}")
            print(f"  R²: {r2:.4f}")

            # Show transformed feature dimensions
            X_transformed = preprocessor.transform(X_test)
            print(f"\n🔄 Transformed feature space:")
            print(f"  Original features: {X_test.shape[1]}")
            print(f"  Transformed features: {X_transformed.shape[1]}")
            print(
                    f"  (Numeric: {len(numeric_features)}, One-Hot Encoded Categorical: {X_transformed.shape[1] - len(numeric_features)})")

            print(f"\n✓ Preprocessing pipeline demonstration complete")
        else:
            print(f"\n⚠ Insufficient clean data ({len(X_clean)} samples) for pipeline demo")
    else:
        print("\n⚠ No target variable or features available for pipeline demo")

except Exception as e:
    logger.error(f"Pipeline demo failed: {e}")
    print(f"✗ Pipeline demo failed: {e}")
    import traceback

    traceback.print_exc()

## Visualization

Create visualizations using finance_ml package functions.

In [ ]:
# Create sector heatmap
try:
    if 'sector' in all_stocks_processed.columns and 'mispricing_score' in all_stocks_processed.columns:
        df_for_viz = all_stocks_processed[all_stocks_processed['mispricing_score'].notna()].copy()
        if len(df_for_viz) > 0:
            fig = create_sector_heatmap(df_for_viz)
            print("\n✓ Sector heatmap created")
        else:
            print("\n⚠ No data with mispricing scores for sector heatmap")
except Exception as e:
    logger.error(f"Sector heatmap failed: {e}")
    print(f"✗ Sector heatmap creation failed: {e}")

# Create interactive prediction plot
try:
    # Remove duplicate columns before plotting
    df_for_plot = all_stocks_processed.copy()
    df_for_plot = df_for_plot.loc[:, ~df_for_plot.columns.duplicated()]

    if 'predicted_target' in df_for_plot.columns and 'price_target' in df_for_plot.columns:
        # Filter to rows with both values
        mask = df_for_plot['predicted_target'].notna() & df_for_plot['price_target'].notna()
        df_for_plot = df_for_plot[mask]

        if len(df_for_plot) > 0:
            fig = create_interactive_prediction_plot(df_for_plot)
            print("✓ Interactive prediction plot created")
        else:
            print("⚠ No data with both predicted and actual targets for plot")
    else:
        print("⚠ Missing required columns for prediction plot")

except Exception as e:
    logger.error(f"Prediction plot failed: {e}")
    print(f"✗ Prediction plot creation failed: {e}")

print("\n" + "=" * 80)
print("Analysis complete!")
print("=" * 80)

In [ ]:
# Visualize dimensionality reduction results
try:
    _transformed = locals().get('transformed_data', None)
    _dim_res = locals().get('dim_reduction_results', None)
    _viz = locals().get('visualizer', None)
    _y_cls = locals().get('y_class_train', None)
    if HAVE_DIM_REDUCTION and isinstance(_transformed, dict) and len(_transformed) > 0:
        print("\n" + "=" * 80)
        print("DIMENSIONALITY REDUCTION VISUALIZATIONS")
        print("=" * 80)

        # Plot explained variance for PCA if available
        if isinstance(_dim_res, dict) and 'PCA' in _dim_res:
            pca_entry = _dim_res.get('PCA', {})
            pca_method = pca_entry.get('fitted_method') if isinstance(pca_entry, dict) else None
            if pca_method is not None and hasattr(pca_method,
                                                  'explained_variance_ratio_') and _viz is not None and hasattr(_viz,
                                                                                                                'plot_explained_variance'):
                _viz.plot_explained_variance(pca_method, "PCA Explained Variance Analysis")

        # Create 2D visualizations for methods that support it
        visualization_methods = {}
        for method_name, X_transformed in _transformed.items():
            try:
                if hasattr(X_transformed, 'shape') and X_transformed.shape[1] >= 2:
                    visualization_methods[method_name] = X_transformed[:, :2]
            except Exception:
                continue

        # Create comparison plots if we have 2D data and a visualizer
        if visualization_methods and _viz is not None and _y_cls is not None and hasattr(_viz,
                                                                                         'plot_component_comparison'):
            _viz.plot_component_comparison(visualization_methods, _y_cls)

        print("Visualization plots generated successfully")
    else:
        print("Dimensionality reduction disabled or not available; skipping.")
except Exception as e:
    logger.error(f"Error in visualization: {e}")
    print(f"Error in visualization: {e}")


## Key Improvements: Proper Preprocessing Pipelines

This notebook now uses the returned `numeric_features` and `categorical_features` lists from `build_features_and_target()` to create proper preprocessing pipelines with:

1. **Separate Transformers**: 
   - `StandardScaler` for numeric features
   - `OneHotEncoder` for categorical features

2. **Benefits**:
   - Proper handling of different feature types
   - Prevents data leakage by fitting transformers only on training data
   - Automatically handles unknown categories in test data
   - Cleaner, more maintainable code

3. **Implementation**:
   - `build_features_and_target()` returns 4 values: `X, y, numeric_features, categorical_features`
   - These lists are used in `ColumnTransformer` for proper preprocessing
   - All model training functions now use this pattern

See the "Advanced Preprocessing Pipeline Demonstration" section above for a working example.

In [ ]:
# Week 1 Enhancement Validation Test Suite
print("=" * 80)
print("WEEK 1 ENHANCEMENT VALIDATION TEST SUITE")
print("=" * 80)


def test_data_quality_validation():
    """Test enhanced data quality validation functionality"""
    print("\n🧪 Testing Data Quality Validation...")

    try:
        # Create test dataset with known issues
        test_data = pd.DataFrame({
            'feature1': [1.0, 2.0, np.inf, 4.0, 5.0],
            'feature2': [10.0, 20.0, 30.0, np.nan, 50.0],
            'feature3': [100.0, -np.inf, 300.0, 400.0, 500.0],
            'Sector': ['Tech', 'Finance', 'Healthcare', 'Tech', 'Finance']
            })

        # Test data quality validation function
        quality_results = validate_financial_data_quality(test_data, "test_region")

        # Validate expected results
        assert quality_results['total_rows'] == 5, "Row count validation failed"
        assert quality_results['infinity_values'] == 2, "Infinity detection failed"  # inf and -inf
        assert quality_results['null_values'] == 1, "Null detection failed"  # 1 NaN
        assert 'data_quality_score' in quality_results, "Quality score missing"

        print("✅ Data quality validation tests passed")
        return True

    except Exception as e:
        print(f"❌ Data quality validation test failed: {e}")
        import traceback
        traceback.print_exc()
        return False


def test_sanitization_monitoring():
    """Test comprehensive sanitization logging functionality"""
    print("\n🧪 Testing Sanitization Monitoring...")

    try:
        # Create test dataset with various data quality issues
        test_data = pd.DataFrame({
            'numeric1': [1.0, 2.0, np.inf, 4.0, 5.0, -np.inf],
            'numeric2': [10.0, 20.0, 1000000.0, np.nan, 50.0, 60.0],  # extreme value
            'numeric3': [100.0, 200.0, 300.0, np.nan, np.nan, 600.0]  # multiple NaN
            })

        # Apply sanitization
        cleaned_data = sanitize_dataframe_with_logging(test_data)

        # Validate sanitization results
        post_inf_count = np.isinf(cleaned_data.select_dtypes(include=[np.number])).sum().sum()
        post_nan_count = cleaned_data.isnull().sum().sum()

        assert post_inf_count == 0, "Infinity values not properly removed"
        assert post_nan_count == 0, "NaN values not properly filled"
        assert cleaned_data.shape == test_data.shape, "Data shape changed unexpectedly"

        print("✅ Sanitization monitoring tests passed")
        return True

    except Exception as e:
        print(f"❌ Sanitization monitoring test failed: {e}")
        import traceback
        traceback.print_exc()
        return False


def test_training_monitoring():
    """Test enhanced training monitoring functionality"""
    print("\n🧪 Testing Training Monitoring...")

    try:
        # Create simple test model and data
        from sklearn.linear_model import LinearRegression
        from sklearn.model_selection import train_test_split

        # Generate test data
        X_test = pd.DataFrame({
            'feature1': np.random.random(100),
            'feature2': np.random.random(100),
            'feature3': np.random.random(100)
            })
        y_test = X_test['feature1'] * 2 + X_test['feature2'] * 3 + np.random.random(100) * 0.1

        X_train, X_val, y_train, y_val = train_test_split(X_test, y_test, test_size=0.3, random_state=42)

        # Create test model
        test_model = LinearRegression()

        # Apply monitoring
        monitoring_results, y_train_pred, y_val_pred = monitor_ensemble_training(
                test_model, X_train, y_train, X_val, y_val, "Test_Model"
                )

        # Validate monitoring results structure
        required_keys = ['model_name', 'timestamp', 'training_time_seconds', 'performance_metrics']
        for key in required_keys:
            assert key in monitoring_results, f"Missing key in monitoring results: {key}"

        # Validate performance metrics
        perf_metrics = monitoring_results['performance_metrics']
        required_perf_keys = ['train_r2', 'test_r2', 'train_mse', 'test_mse']
        for key in required_perf_keys:
            assert key in perf_metrics, f"Missing performance metric: {key}"

        print("✅ Training monitoring tests passed")
        return True

    except Exception as e:
        print(f"❌ Training monitoring test failed: {e}")
        import traceback
        traceback.print_exc()
        return False


def test_pipeline_validation():
    """Test early pipeline validation functionality"""
    print("\n🧪 Testing Pipeline Validation...")

    try:
        # Create test dataset that mimics financial data
        test_financial_data = pd.DataFrame({
            'p_e_ntm': [15.0, 25.0, -5.0, 1500.0, 20.0],  # negative and extreme values
            'market_cap': [1000000, 2000000, -500000, 5000000, 1500000],  # negative value
            'last_price': [50.0, 100.0, 0.0, 200.0, 75.0],  # zero price
            'price_target': [60.0, 120.0, 50.0, 180.0, 80.0],
            'sector': ['Technology', 'Healthcare', 'Finance', 'Technology', 'Energy'],
            'ticker': ['AAPL', 'JNJ', 'JPM', 'MSFT', 'XOM'],
            'next_earnings_days': [30, 45, -10, 60, np.nan]  # past date and NaN
            })

        # Test pipeline validation function
        validation_results = perform_early_pipeline_validation(test_financial_data)

        # Validate results structure
        required_keys = ['validation_score', 'total_checks', 'passed_checks', 'warnings', 'recommendations']
        for key in required_keys:
            assert key in validation_results, f"Missing key in validation results: {key}"

        # Check that warnings were detected for problematic data
        assert len(validation_results['warnings']) > 0, "Expected warnings for problematic test data"
        assert 'validation_score' in validation_results, "Validation score missing"
        assert 0 <= validation_results['validation_score'] <= 1, "Validation score out of range"

        print("✅ Pipeline validation tests passed")
        return True

    except Exception as e:
        print(f"❌ Pipeline validation test failed: {e}")
        import traceback
        traceback.print_exc()
        return False


# Execute validation test suite
print("\nExecuting Week 1 Enhancement Validation Tests...")
print("-" * 60)

test_results = {
    'data_quality_validation': test_data_quality_validation(),
    'sanitization_monitoring': test_sanitization_monitoring(),
    'training_monitoring': test_training_monitoring(),
    'pipeline_validation': test_pipeline_validation()
    }

# Summary results
print("\n" + "=" * 60)
print("VALIDATION TEST RESULTS SUMMARY")
print("=" * 60)

passed_tests = sum(test_results.values())
total_tests = len(test_results)
success_rate = (passed_tests / total_tests) * 100

for test_name, result in test_results.items():
    status = "✅ PASSED" if result else "❌ FAILED"
    print(f"{test_name.replace('_', ' ').title():<30}: {status}")

print("-" * 60)
print(f"Overall Success Rate: {passed_tests}/{total_tests} ({success_rate:.1f}%)")

if success_rate >= 75:
    print("🎉 Week 1 enhancements validation SUCCESSFUL!")
    print("✅ Ready for CI/CD integration")
else:
    print("⚠️ Some Week 1 enhancements need attention")
    print("🔧 Review failed tests before proceeding")

print("=" * 60)


## Per-Sector Regression, Quantile Bands, Stacking, and Excel Export (Implemented from Examine_theml_finance_model_v8_2.ipynb_f.md)

This section implements key enhancements from the planning document:
- Per-sector regression metrics
- Quantile regression by sector for uncertainty bands
- Stacking ensemble by sector
- Excel export with predictions and summaries

All steps are guarded to keep the notebook robust on small demo datasets.

In [ ]:
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd

try:
    from finance_ml import (
        train_and_evaluate_regression_by_sector,
        train_quantile_regression_by_sector,
        predict_quantile_regression,
        train_stacking_ensemble_by_sector,
        export_predictions_to_excel,
        )

    HAVE_ENHANCED_MODELS = True
except Exception as e:
    print(f"⚠ Enhanced modeling imports unavailable: {e}")
    HAVE_ENHANCED_MODELS = False

# Ensure output directory
try:
    output_dir = Path(config.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
except Exception:
    output_dir = Path("outputs")
    output_dir.mkdir(parents=True, exist_ok=True)

# Remove duplicate columns first
df_enhanced = all_stocks_processed.loc[:, ~all_stocks_processed.columns.duplicated()].copy()

# --- Per-sector regression metrics ---
if HAVE_ENHANCED_MODELS:
    try:
        print("\n" + "=" * 80)
        print("PER-SECTOR REGRESSION METRICS")
        print("=" * 80)
        sector_metrics = train_and_evaluate_regression_by_sector(df_enhanced, output_dir)
        display_cols = [c for c in ["sector", "n_train", "n_test", "mae", "rmse", "r2"] if c in sector_metrics.columns]
        if len(sector_metrics) > 0:
            print(sector_metrics[display_cols].to_string(index=False))
        else:
            print("(No sectors with sufficient data)")
    except Exception as e:
        print(f"⚠ Per-sector regression metrics step skipped: {e}")

# --- Quantile regression by sector (uncertainty bands) ---
quantile_models_by_sector = {}
if HAVE_ENHANCED_MODELS:
    try:
        # Determine target column
        target_col = None
        for cand in ["price_target", "price_target_median"]:
            if cand in df_enhanced.columns:
                target_col = cand
                break

        if target_col is None:
            raise ValueError("No target column found")

        # Select numeric feature columns only
        numeric_cols = [
            c for c in df_enhanced.columns
            if pd.api.types.is_numeric_dtype(df_enhanced[c])
            ]
        blacklist = {target_col, "predicted_target", "mispricing_score"}
        feature_cols = [c for c in numeric_cols if c not in blacklist]

        if len(feature_cols) < 3:
            raise ValueError(f"Insufficient numeric features ({len(feature_cols)}) for quantile regression")

        # Check we have enough clean data
        required_cols = feature_cols + [target_col]
        if 'sector' in df_enhanced.columns:
            required_cols.append('sector')

        df_clean = df_enhanced[required_cols].dropna()

        if len(df_clean) < 50:
            raise ValueError(f"Insufficient clean data ({len(df_clean)} rows) for quantile regression")

        print("\n" + "=" * 80)
        print("QUANTILE REGRESSION BY SECTOR (q10/q50/q90)")
        print("=" * 80)

        quantile_models_by_sector = train_quantile_regression_by_sector(
                df_enhanced, feature_cols, target_col, quantiles=[0.1, 0.5, 0.9]
                )

        # Generate quantile predictions per sector
        prediction_count = 0
        for sector, model in quantile_models_by_sector.items():
            sec_mask = (df_enhanced.get("sector") == sector) if "sector" in df_enhanced.columns else pd.Series(False,
                                                                                                               index=df_enhanced.index)
            if sec_mask.sum() == 0:
                continue

            X_sec = df_enhanced.loc[sec_mask, feature_cols].dropna()
            if len(X_sec) == 0:
                continue

            preds_df = predict_quantile_regression(model, X_sec, quantiles=[0.1, 0.5, 0.9])

            # Assign predictions back to main frame
            for col in preds_df.columns:
                df_enhanced.loc[X_sec.index, col] = preds_df[col].values
            prediction_count += len(X_sec)

        if prediction_count > 0:
            print(
                    f"✓ Quantile bands added for {prediction_count} stocks across {len(quantile_models_by_sector)} sectors")
        else:
            print("⚠ No quantile predictions generated")

    except Exception as e:
        print(f"⚠ Quantile regression step skipped: {e}")

# --- Stacking ensemble by sector ---
stacking_models_by_sector = {}
if HAVE_ENHANCED_MODELS:
    try:
        if 'feature_cols' in locals() and 'target_col' in locals():
            cols_needed = [c for c in (feature_cols + [target_col, 'sector']) if c in df_enhanced.columns]
            df_stack = df_enhanced[cols_needed].dropna()

            if len(df_stack) >= 100:  # Increased threshold for stability
                print("\n" + "=" * 80)
                print("STACKING ENSEMBLE BY SECTOR")
                print("=" * 80)
                stacking_models_by_sector = train_stacking_ensemble_by_sector(
                        df_stack, feature_cols, target_col
                        )
                print(f"✓ Trained stacking ensembles for {len(stacking_models_by_sector)} sectors")
            else:
                print(f"⚠ Skipping stacking ensemble — insufficient clean rows: {len(df_stack)} (need 100+)")
        else:
            print("⚠ Skipping stacking ensemble — features/target not available")
    except Exception as e:
        print(f"⚠ Stacking ensemble step skipped: {e}")

# --- Excel export ---
if HAVE_ENHANCED_MODELS:
    try:
        if 'mispricing_score' in df_enhanced.columns and df_enhanced['mispricing_score'].notna().sum() > 0:
            ts = datetime.now().strftime("%Y%m%d_%H%M%S")
            excel_path = output_dir / f"Stock_Prediction_Analysis_Report_{ts}.xlsx"
            export_predictions_to_excel(df_enhanced, excel_path, include_summary=True)
            print(f"\n✓ Exported predictions and summaries to Excel: {excel_path}")
        else:
            print("⚠ Excel export skipped — mispricing_score not found; run valuation analysis first")
    except ImportError as e:
        print(f"⚠ Excel export skipped (engine missing): {e}")
    except Exception as e:
        print(f"⚠ Excel export failed: {e}")

# Update main dataframe with enhancements
all_stocks_processed = df_enhanced

## Reporting: Export predictions CSV and run summary JSON

In [ ]:
# Export artifacts to outputs/ for downstream analysis and automation
try:
    from pathlib import Path
    import os
    import json

    # Resolve output directory
    out_dir = Path(config.output_dir) if 'config' in locals() and hasattr(config, 'output_dir') else Path('outputs')
    out_dir.mkdir(parents=True, exist_ok=True)

    # Export predictions CSV if available
    exported_csv_path = out_dir / 'regression_predictions.csv'
    if 'predicted_target' in all_stocks_processed.columns:
        cols = [c for c in ['ticker', 'sector', 'region', 'last_price', 'predicted_target', 'mispricing_score']
                if c in all_stocks_processed.columns]
        if cols:
            df_export = all_stocks_processed.loc[all_stocks_processed['predicted_target'].notna(), cols].copy()
            if len(df_export) > 0:
                df_export.to_csv(exported_csv_path, index=False)
                print(f"\n✓ Exported predictions CSV: {exported_csv_path}")
            else:
                print("\n⚠ No rows with predictions to export")
        else:
            print("\n⚠ Required columns missing for predictions export")
    else:
        print("\n⚠ 'predicted_target' column not found; skipping predictions CSV export")

    # Build a compact run summary JSON
    run_summary = {}
    # Prefer metrics from regression_results if available
    if isinstance(globals().get('regression_results'), dict):
        for k in ['mae', 'rmse', 'r2']:
            if k in regression_results and regression_results[k] is not None:
                try:
                    run_summary[k] = float(regression_results[k])
                except Exception:
                    pass
    # Fallback: try metrics from advanced demo if present
    for name in ['mae', 'rmse', 'r2']:
        if name not in run_summary and name in globals():
            try:
                run_summary[name] = float(globals()[name])
            except Exception:
                pass

    # Meta information
    run_summary['model_version'] = os.getenv('MODEL_VERSION', 'v8_3')
    run_summary['rows_total'] = int(len(all_stocks_processed)) if 'all_stocks_processed' in locals() else 0
    run_summary['rows_scored'] = int(all_stocks_processed['predicted_target'].notna().sum()) \
        if 'predicted_target' in all_stocks_processed.columns else 0

    with (out_dir / 'run_summary.json').open('w', encoding='utf-8') as f:
        json.dump(run_summary, f, indent=2)
    print(f"✓ Wrote run_summary.json in {out_dir}")

except Exception as e:
    import traceback

    print(f"✗ Reporting export failed: {e}")
    traceback.print_exc()


In [ ]:
# ============================================================================
# COMPREHENSIVE SUMMARY STATISTICS
# ============================================================================
print("\n" + "=" * 80)
print("COMPREHENSIVE SUMMARY STATISTICS FOR ALL_STOCKS DATAFRAME")
print("=" * 80)

try:
    # 1. Basic Dataset Overview
    print("\n" + "-" * 80)
    print("1. DATASET OVERVIEW")
    print("-" * 80)
    print(f"Total Stocks: {len(all_stocks_processed):,}")
    print(f"Total Columns: {len(all_stocks_processed.columns)}")
    print(f"Memory Usage: {all_stocks_processed.memory_usage(deep=True).sum() / 1024 ** 2:.2f} MB")

    # 2. Regional Distribution
    if 'region' in all_stocks_processed.columns:
        print("\n" + "-" * 80)
        print("2. REGIONAL DISTRIBUTION")
        print("-" * 80)
        region_stats = all_stocks_processed['region'].value_counts()
        region_pct = (region_stats / len(all_stocks_processed) * 100).round(2)
        region_df = pd.DataFrame({
            'Count': region_stats,
            'Percentage': region_pct
            })
        print(region_df.to_string())

    # 3. Sector Distribution
    if 'sector' in all_stocks_processed.columns:
        print("\n" + "-" * 80)
        print("3. SECTOR DISTRIBUTION")
        print("-" * 80)
        sector_stats = all_stocks_processed['sector'].value_counts()
        sector_pct = (sector_stats / len(all_stocks_processed) * 100).round(2)
        sector_df = pd.DataFrame({
            'Count': sector_stats,
            'Percentage': sector_pct
            })
        print(sector_df.head(10).to_string())

    # 4. Key Financial Metrics Summary
    print("\n" + "-" * 80)
    print("4. KEY FINANCIAL METRICS SUMMARY")
    print("-" * 80)

    key_metrics = ['last_price', 'market_cap', 'p_e_ntm', 'ev_over_ebitda',
                   'net_debt_over_ebitda', 'revenue_growth_3y', 'ebitda_margin']
    available_metrics = [m for m in key_metrics if m in all_stocks_processed.columns]

    if available_metrics:
        stats_df = all_stocks_processed[available_metrics].describe()
        print(stats_df.to_string())

        # Additional statistics
        print("\n📊 Additional Statistics:")
        for col in available_metrics:
            if pd.api.types.is_numeric_dtype(all_stocks_processed[col]):
                non_null = all_stocks_processed[col].notna().sum()
                null_pct = (all_stocks_processed[col].isna().sum() / len(all_stocks_processed) * 100)
                print(f"  {col}: {non_null:,} non-null ({100 - null_pct:.1f}% coverage)")

    # 5. Data Quality Metrics
    print("\n" + "-" * 80)
    print("5. DATA QUALITY METRICS")
    print("-" * 80)

    # Calculate missing data percentage
    missing_data = all_stocks_processed.isnull().sum()
    missing_pct = (missing_data / len(all_stocks_processed) * 100).round(2)
    missing_df = pd.DataFrame({
        'Missing_Count': missing_data,
        'Missing_Pct': missing_pct
        }).sort_values('Missing_Count', ascending=False)

    print("Top 15 columns with missing data:")
    print(missing_df[missing_df['Missing_Count'] > 0].head(15).to_string())

    # 6. Target Variable Distribution (if exists)
    target_cols = ['price_target', 'predicted_target', 'mispricing_score']
    available_targets = [t for t in target_cols if t in all_stocks_processed.columns]

    if available_targets:
        print("\n" + "-" * 80)
        print("6. TARGET VARIABLE STATISTICS")
        print("-" * 80)

        for target in available_targets:
            if all_stocks_processed[target].notna().sum() > 0:
                print(f"\n{target.upper()}:")
                print(f"  Count: {all_stocks_processed[target].notna().sum():,}")
                print(f"  Mean: {all_stocks_processed[target].mean():.2f}")
                print(f"  Median: {all_stocks_processed[target].median():.2f}")
                print(f"  Std Dev: {all_stocks_processed[target].std():.2f}")
                print(f"  Min: {all_stocks_processed[target].min():.2f}")
                print(f"  Max: {all_stocks_processed[target].max():.2f}")

    # 7. Sector-wise Key Metrics
    if 'sector' in all_stocks_processed.columns and 'market_cap' in all_stocks_processed.columns:
        print("\n" + "-" * 80)
        print("7. SECTOR-WISE AGGREGATIONS")
        print("-" * 80)

        sector_agg = all_stocks_processed.groupby('sector').agg({
            'market_cap': ['count', 'mean', 'median'],
            'last_price': ['mean', 'median'] if 'last_price' in all_stocks_processed.columns else ['count']
            })

        sector_agg.columns = ['_'.join(col).strip() for col in sector_agg.columns]
        print(sector_agg.sort_values(sector_agg.columns[0], ascending=False).head(10).to_string())

    print("\n" + "=" * 80)
    print("✓ Summary statistics completed successfully")
    print("=" * 80)

except Exception as e:
    print(f"\n✗ Error generating summary statistics: {e}")
    import traceback

    traceback.print_exc()

In [ ]:
# ============================================================================
# DISTRIBUTION VISUALIZATIONS (Matplotlib/Seaborn)
# ============================================================================
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for better-looking plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("\n" + "=" * 80)
print("CREATING DISTRIBUTION VISUALIZATIONS")
print("=" * 80)

try:
    # 1. Regional Distribution Bar Chart
    if 'region' in all_stocks_processed.columns:
        region_counts = all_stocks_processed['region'].value_counts()
        if len(region_counts) > 0:
            fig, ax = plt.subplots(1, 1, figsize=(10, 6))
            sns.barplot(x=region_counts.index, y=region_counts.values, ax=ax, palette='viridis')
            ax.set_title('Stock Distribution by Region', fontsize=16, fontweight='bold')
            ax.set_xlabel('Region', fontsize=12)
            ax.set_ylabel('Count', fontsize=12)
            if len(ax.containers) > 0:
                ax.bar_label(ax.containers[0], fmt='%d')
            plt.tight_layout()
            plt.show()
            print("✓ Regional distribution chart created")
        else:
            print("⚠ No region data available for visualization")

    # 2. Sector Distribution Bar Chart (Top 10)
    if 'sector' in all_stocks_processed.columns:
        sector_counts = all_stocks_processed['sector'].value_counts().head(10)
        if len(sector_counts) > 0:
            fig, ax = plt.subplots(1, 1, figsize=(12, 8))
            sns.barplot(y=sector_counts.index, x=sector_counts.values, ax=ax, palette='coolwarm')
            ax.set_title('Top 10 Sectors by Stock Count', fontsize=16, fontweight='bold')
            ax.set_xlabel('Count', fontsize=12)
            ax.set_ylabel('Sector', fontsize=12)
            if len(ax.containers) > 0:
                ax.bar_label(ax.containers[0], fmt='%d')
            plt.tight_layout()
            plt.show()
            print("✓ Sector distribution chart created")
        else:
            print("⚠ No sector data available for visualization")

    # 3. Key Financial Metrics Distributions (Histograms with KDE)
    key_metrics = ['last_price', 'market_cap', 'p_e_ntm', 'ev_over_ebitda']
    available_metrics = [m for m in key_metrics if m in all_stocks_processed.columns]

    if available_metrics:
        n_metrics = len(available_metrics)
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        axes = axes.flatten()

        for idx, metric in enumerate(available_metrics[:4]):
            data = all_stocks_processed[metric].dropna()
            if len(data) > 0:
                # Handle outliers by using log scale for large values
                if data.max() > 1000:
                    data_plot = np.log10(data + 1)
                    xlabel = f'log10({metric})'
                else:
                    data_plot = data
                    xlabel = metric

                sns.histplot(data_plot, kde=True, ax=axes[idx], color='steelblue', bins=30)
                axes[idx].set_title(f'Distribution of {metric}', fontsize=12, fontweight='bold')
                axes[idx].set_xlabel(xlabel, fontsize=10)
                axes[idx].set_ylabel('Frequency', fontsize=10)

        # Hide unused subplots
        for idx in range(n_metrics, 4):
            axes[idx].set_visible(False)

        plt.tight_layout()
        plt.show()
        print(f"✓ Distribution histograms created for {n_metrics} metrics")

    # 4. Box Plots for Key Metrics by Region
    if 'region' in all_stocks_processed.columns and 'market_cap' in all_stocks_processed.columns:
        fig, axes = plt.subplots(1, 2, figsize=(15, 6))

        # Market Cap by Region
        data_mktcap = all_stocks_processed[all_stocks_processed['market_cap'].notna()]
        if len(data_mktcap) > 0:
            sns.boxplot(data=data_mktcap, x='region', y='market_cap', ax=axes[0], palette='Set2')
            axes[0].set_yscale('log')
            axes[0].set_title('Market Cap Distribution by Region (Log Scale)', fontsize=12, fontweight='bold')
            axes[0].set_xlabel('Region', fontsize=10)
            axes[0].set_ylabel('Market Cap (log scale)', fontsize=10)

        # Last Price by Region
        if 'last_price' in all_stocks_processed.columns:
            data_price = all_stocks_processed[all_stocks_processed['last_price'].notna()]
            if len(data_price) > 0:
                sns.boxplot(data=data_price, x='region', y='last_price', ax=axes[1], palette='Set3')
                axes[1].set_title('Stock Price Distribution by Region', fontsize=12, fontweight='bold')
                axes[1].set_xlabel('Region', fontsize=10)
                axes[1].set_ylabel('Last Price', fontsize=10)

        plt.tight_layout()
        plt.show()
        print("✓ Box plots by region created")

    # 5. Violin Plots for P/E Ratio by Sector (Top 5 sectors)
    if 'sector' in all_stocks_processed.columns and 'p_e_ntm' in all_stocks_processed.columns:
        top_sectors = all_stocks_processed['sector'].value_counts().head(5).index
        data_pe = all_stocks_processed[
            (all_stocks_processed['sector'].isin(top_sectors)) &
            (all_stocks_processed['p_e_ntm'].notna()) &
            (all_stocks_processed['p_e_ntm'] > 0) &
            (all_stocks_processed['p_e_ntm'] < 100)  # Filter extreme outliers
            ]

        if len(data_pe) > 10:
            fig, ax = plt.subplots(1, 1, figsize=(14, 6))
            sns.violinplot(data=data_pe, x='sector', y='p_e_ntm', ax=ax, palette='muted')
            ax.set_title('P/E Ratio Distribution by Top 5 Sectors', fontsize=14, fontweight='bold')
            ax.set_xlabel('Sector', fontsize=11)
            ax.set_ylabel('P/E Ratio (NTM)', fontsize=11)
            plt.xticks(rotation=45, ha='right')
            plt.tight_layout()
            plt.show()
            print("✓ Violin plots for P/E by sector created")

    print("\n" + "=" * 80)
    print("✓ All distribution visualizations completed")
    print("=" * 80)

except Exception as e:
    print(f"\n✗ Error creating distribution visualizations: {e}")
    import traceback

    traceback.print_exc()

In [ ]:
# ============================================================================
# CORRELATION ANALYSIS AND HEATMAPS
# ============================================================================
print("\n" + "=" * 80)
print("CREATING CORRELATION ANALYSIS VISUALIZATIONS")
print("=" * 80)

try:
    # Select key financial metrics for correlation analysis
    correlation_metrics = [
        'last_price', 'market_cap', 'enterprise_value',
        'p_e_ntm', 'p_b', 'ev_over_ebitda', 'net_debt_over_ebitda',
        'revenue_growth_1y', 'revenue_growth_3y',
        'ebitda_margin', 'net_margin', 'roe'
        ]

    # Filter to available columns
    available_corr_metrics = [m for m in correlation_metrics if m in all_stocks_processed.columns]

    if len(available_corr_metrics) >= 3:
        # 1. Correlation Matrix Heatmap (Pearson)
        corr_data = all_stocks_processed[available_corr_metrics].dropna()

        if len(corr_data) > 10:
            fig, ax = plt.subplots(1, 1, figsize=(14, 12))

            # Calculate Pearson correlation
            corr_matrix = corr_data.corr(method='pearson')

            # Create heatmap
            sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
                        center=0, vmin=-1, vmax=1, square=True, ax=ax,
                        cbar_kws={'label': 'Correlation Coefficient'})
            ax.set_title('Correlation Matrix of Financial Metrics (Pearson)',
                         fontsize=14, fontweight='bold', pad=20)
            plt.xticks(rotation=45, ha='right')
            plt.yticks(rotation=0)
            plt.tight_layout()
            plt.show()
            print(f"✓ Correlation heatmap created ({len(available_corr_metrics)} metrics)")

            # 2. Spearman Correlation for non-linear relationships
            fig, ax = plt.subplots(1, 1, figsize=(14, 12))
            corr_matrix_spearman = corr_data.corr(method='spearman')

            sns.heatmap(corr_matrix_spearman, annot=True, fmt='.2f', cmap='viridis',
                        center=0, vmin=-1, vmax=1, square=True, ax=ax,
                        cbar_kws={'label': 'Spearman Rank Correlation'})
            ax.set_title('Correlation Matrix of Financial Metrics (Spearman)',
                         fontsize=14, fontweight='bold', pad=20)
            plt.xticks(rotation=45, ha='right')
            plt.yticks(rotation=0)
            plt.tight_layout()
            plt.show()
            print(f"✓ Spearman correlation heatmap created")

            # 3. Top Correlations Summary
            print("\n📊 Strongest Positive Correlations (excluding diagonal):")
            corr_pairs = []
            for i in range(len(corr_matrix.columns)):
                for j in range(i + 1, len(corr_matrix.columns)):
                    corr_pairs.append({
                        'Metric 1': corr_matrix.columns[i],
                        'Metric 2': corr_matrix.columns[j],
                        'Correlation': corr_matrix.iloc[i, j]
                        })

            corr_df = pd.DataFrame(corr_pairs).sort_values('Correlation', ascending=False)
            print(corr_df.head(10).to_string(index=False))

            print("\n📊 Strongest Negative Correlations:")
            print(corr_df.tail(10).to_string(index=False))

    # 4. Scatter Plot Matrix for Top 5 Metrics
    top_metrics_for_scatter = ['market_cap', 'p_e_ntm', 'ev_over_ebitda', 'revenue_growth_1y', 'ebitda_margin']
    available_scatter = [m for m in top_metrics_for_scatter if m in all_stocks_processed.columns]

    if len(available_scatter) >= 3:
        scatter_data = all_stocks_processed[available_scatter].dropna()

        # Sample if too large for performance
        if len(scatter_data) > 1000:
            scatter_data = scatter_data.sample(1000, random_state=42)

        if len(scatter_data) > 20:
            print(f"\n📊 Creating pair plot for {len(available_scatter)} metrics (sample size: {len(scatter_data)})...")

            # Create pairplot
            pairplot = sns.pairplot(scatter_data, diag_kind='kde', plot_kws={'alpha': 0.6},
                                    corner=False, height=2.5)
            pairplot.fig.suptitle('Pair Plot of Key Financial Metrics',
                                  fontsize=16, fontweight='bold', y=1.01)
            plt.tight_layout()
            plt.show()
            print("✓ Scatter plot matrix created")

    # 5. Correlation with Target Variable (if available)
    if 'price_target' in all_stocks_processed.columns:
        target_corr_metrics = [m for m in available_corr_metrics if m != 'price_target']
        if target_corr_metrics:
            target_corr_data = all_stocks_processed[target_corr_metrics + ['price_target']].dropna()

            if len(target_corr_data) > 10:
                target_correlations = target_corr_data.corr()['price_target'].drop('price_target').sort_values(
                        ascending=False)

                fig, ax = plt.subplots(1, 1, figsize=(10, 8))
                sns.barplot(y=target_correlations.index, x=target_correlations.values,
                            ax=ax, palette='RdYlGn')
                ax.set_title('Feature Correlation with Price Target',
                             fontsize=14, fontweight='bold')
                ax.set_xlabel('Correlation Coefficient', fontsize=11)
                ax.set_ylabel('Feature', fontsize=11)
                ax.axvline(x=0, color='black', linestyle='--', linewidth=0.8)
                plt.tight_layout()
                plt.show()
                print("✓ Target correlation chart created")

    print("\n" + "=" * 80)
    print("✓ All correlation analysis completed")
    print("=" * 80)

except Exception as e:
    print(f"\n✗ Error creating correlation visualizations: {e}")
    import traceback

    traceback.print_exc()

## Interactive Visualizations (Plotly)

Create interactive charts for exploration and dashboarding.

In [ ]:
# ============================================================================
# INTERACTIVE VISUALIZATIONS (Plotly)
# ============================================================================
import plotly.express as px
import plotly.graph_objects as go

print("\n" + "=" * 80)
print("CREATING INTERACTIVE VISUALIZATIONS (PLOTLY)")
print("=" * 80)

try:
    # 1. Interactive Scatter: Market Cap vs P/E by Sector
    if all(['market_cap' in all_stocks_processed.columns,
            'p_e_ntm' in all_stocks_processed.columns,
            'sector' in all_stocks_processed.columns]):

        scatter_data = all_stocks_processed[
            (all_stocks_processed['market_cap'].notna()) &
            (all_stocks_processed['p_e_ntm'].notna()) &
            (all_stocks_processed['p_e_ntm'] > 0) &
            (all_stocks_processed['p_e_ntm'] < 100)
            ].copy()

        if len(scatter_data) > 10:
            fig = px.scatter(
                    scatter_data,
                    x='market_cap',
                    y='p_e_ntm',
                    color='sector',
                    size='market_cap',
                    hover_data=['ticker', 'name'] if 'ticker' in scatter_data.columns else None,
                    title='Market Cap vs P/E Ratio by Sector (Interactive)',
                    labels={'market_cap': 'Market Capitalization', 'p_e_ntm': 'P/E Ratio (NTM)'},
                    log_x=True
                    )
            fig.update_layout(height=600, showlegend=True)
            fig.show()
            print("✓ Interactive scatter plot (Market Cap vs P/E) created")

    # 2. Interactive Sunburst: Regional and Sector Breakdown
    if all(['region' in all_stocks_processed.columns,
            'sector' in all_stocks_processed.columns]):

        # Aggregate data for sunburst
        sunburst_data = all_stocks_processed.groupby(['region', 'sector']).size().reset_index(name='count')

        if len(sunburst_data) > 0:
            fig = px.sunburst(
                    sunburst_data,
                    path=['region', 'sector'],
                    values='count',
                    title='Stock Distribution: Region → Sector Hierarchy',
                    color='count',
                    color_continuous_scale='Viridis'
                    )
            fig.update_layout(height=700)
            fig.show()
            print("✓ Interactive sunburst chart created")

    # 3. Interactive Box Plot: Valuation Metrics by Region
    if 'region' in all_stocks_processed.columns:
        valuation_metrics = ['p_e_ntm', 'p_b', 'ev_over_ebitda']
        available_val = [m for m in valuation_metrics if m in all_stocks_processed.columns]

        if available_val:
            metric = available_val[0]
            box_data = all_stocks_processed[
                (all_stocks_processed[metric].notna()) &
                (all_stocks_processed[metric] > 0) &
                (all_stocks_processed[metric] < 100)
                ]

            if len(box_data) > 20:
                fig = px.box(
                        box_data,
                        x='region',
                        y=metric,
                        color='region',
                        title=f'{metric.upper()} Distribution by Region (Interactive)',
                        labels={metric: metric.upper(), 'region': 'Region'}
                        )
                fig.update_layout(height=500, showlegend=False)
                fig.show()
                print(f"✓ Interactive box plot ({metric} by region) created")

    # 4. Interactive Bar Chart: Top Stocks by Market Cap
    if 'market_cap' in all_stocks_processed.columns:
        top_stocks = all_stocks_processed.nlargest(20, 'market_cap')

        if len(top_stocks) > 0:
            fig = px.bar(
                    top_stocks,
                    x='ticker' if 'ticker' in top_stocks.columns else top_stocks.index,
                    y='market_cap',
                    color='sector' if 'sector' in top_stocks.columns else None,
                    title='Top 20 Stocks by Market Capitalization',
                    labels={'market_cap': 'Market Cap', 'ticker': 'Stock Ticker'},
                    hover_data=['name', 'region'] if all(
                            [c in top_stocks.columns for c in ['name', 'region']]) else None
                    )
            fig.update_layout(height=500, xaxis_tickangle=-45)
            fig.show()
            print("✓ Interactive bar chart (Top 20 by Market Cap) created")

    # 5. Interactive 3D Scatter: Market Cap, P/E, and Revenue Growth
    if all(['market_cap' in all_stocks_processed.columns,
            'p_e_ntm' in all_stocks_processed.columns,
            'revenue_growth_1y' in all_stocks_processed.columns,
            'sector' in all_stocks_processed.columns]):

        scatter_3d_data = all_stocks_processed[
            (all_stocks_processed['market_cap'].notna()) &
            (all_stocks_processed['p_e_ntm'].notna()) &
            (all_stocks_processed['revenue_growth_1y'].notna()) &
            (all_stocks_processed['p_e_ntm'] > 0) &
            (all_stocks_processed['p_e_ntm'] < 100)
            ].copy()

        # Sample for performance
        if len(scatter_3d_data) > 500:
            scatter_3d_data = scatter_3d_data.sample(500, random_state=42)

        if len(scatter_3d_data) > 20:
            fig = px.scatter_3d(
                    scatter_3d_data,
                    x='market_cap',
                    y='p_e_ntm',
                    z='revenue_growth_1y',
                    color='sector',
                    size='market_cap',
                    hover_data=['ticker'] if 'ticker' in scatter_3d_data.columns else None,
                    title='3D Analysis: Market Cap, P/E, and Revenue Growth',
                    labels={
                        'market_cap': 'Market Cap',
                        'p_e_ntm': 'P/E Ratio',
                        'revenue_growth_1y': 'Revenue Growth (1Y)'
                        },
                    log_x=True
                    )
            fig.update_layout(height=700)
            fig.show()
            print("✓ Interactive 3D scatter plot created")

    # 6. Interactive Treemap: Market Cap Distribution by Sector and Region
    if all(['sector' in all_stocks_processed.columns,
            'region' in all_stocks_processed.columns,
            'market_cap' in all_stocks_processed.columns]):

        treemap_data = all_stocks_processed[all_stocks_processed['market_cap'].notna()].copy()

        if len(treemap_data) > 10:
            fig = px.treemap(
                    treemap_data,
                    path=['region', 'sector'],
                    values='market_cap',
                    title='Market Capitalization Distribution: Region → Sector',
                    color='market_cap',
                    color_continuous_scale='Blues'
                    )
            fig.update_layout(height=700)
            fig.show()
            print("✓ Interactive treemap created")

    # 7. Time Series of Stock Predictions (if available)
    if all(['predicted_target' in all_stocks_processed.columns,
            'last_price' in all_stocks_processed.columns,
            'ticker' in all_stocks_processed.columns]):

        pred_data = all_stocks_processed[
            (all_stocks_processed['predicted_target'].notna()) &
            (all_stocks_processed['last_price'].notna())
            ].head(30)

        if len(pred_data) > 5:
            fig = go.Figure()

            fig.add_trace(go.Bar(
                    x=pred_data['ticker'],
                    y=pred_data['last_price'],
                    name='Current Price',
                    marker_color='lightblue'
                    ))

            fig.add_trace(go.Bar(
                    x=pred_data['ticker'],
                    y=pred_data['predicted_target'],
                    name='Predicted Target',
                    marker_color='coral'
                    ))

            fig.update_layout(
                    title='Current Price vs Predicted Target (Sample)',
                    xaxis_title='Stock Ticker',
                    yaxis_title='Price',
                    barmode='group',
                    height=500,
                    xaxis_tickangle=-45
                    )
            fig.show()
            print("✓ Interactive grouped bar chart (Price vs Target) created")

    print("\n" + "=" * 80)
    print("✓ All interactive visualizations completed")
    print("=" * 80)

except Exception as e:
    print(f"\n✗ Error creating interactive visualizations: {e}")
    import traceback

    traceback.print_exc()

In [ ]:
# ============================================================================
# SECTOR AND REGION ANALYSIS VISUALIZATIONS
# ============================================================================
print("\n" + "=" * 80)
print("CREATING SECTOR AND REGION ANALYSIS VISUALIZATIONS")
print("=" * 80)

try:
    # 1. Sector Performance Heatmap by Region
    if all(['sector' in all_stocks_processed.columns,
            'region' in all_stocks_processed.columns,
            'market_cap' in all_stocks_processed.columns]):

        # Aggregate market cap by sector and region
        pivot_data = all_stocks_processed.pivot_table(
                values='market_cap',
                index='sector',
                columns='region',
                aggfunc='sum',
                fill_value=0
                )

        if len(pivot_data) > 0:
            fig, ax = plt.subplots(1, 1, figsize=(12, 10))
            sns.heatmap(pivot_data / 1e9, annot=True, fmt='.1f', cmap='YlOrRd',
                        ax=ax, cbar_kws={'label': 'Total Market Cap (Billions)'})
            ax.set_title('Total Market Capitalization by Sector and Region',
                         fontsize=14, fontweight='bold')
            ax.set_xlabel('Region', fontsize=12)
            ax.set_ylabel('Sector', fontsize=12)
            plt.tight_layout()
            plt.show()
            print("✓ Sector-Region market cap heatmap created")

    # 2. Average Valuation Metrics by Sector
    if 'sector' in all_stocks_processed.columns:
        valuation_cols = ['p_e_ntm', 'p_b', 'ev_over_ebitda']
        available_vals = [c for c in valuation_cols if c in all_stocks_processed.columns]

        if available_vals:
            sector_val = all_stocks_processed.groupby('sector')[available_vals].median().dropna()

            if len(sector_val) > 0:
                fig, ax = plt.subplots(1, 1, figsize=(14, 8))
                sector_val.plot(kind='barh', ax=ax, width=0.8)
                ax.set_title('Median Valuation Metrics by Sector',
                             fontsize=14, fontweight='bold')
                ax.set_xlabel('Median Value', fontsize=12)
                ax.set_ylabel('Sector', fontsize=12)
                ax.legend(title='Metrics', bbox_to_anchor=(1.05, 1), loc='upper left')
                plt.tight_layout()
                plt.show()
                print("✓ Sector valuation metrics chart created")

    # 3. Regional Stock Count and Average Market Cap
    if all(['region' in all_stocks_processed.columns,
            'market_cap' in all_stocks_processed.columns]):

        regional_stats = all_stocks_processed.groupby('region').agg({
            'market_cap': ['count', 'mean', 'sum']
            })
        regional_stats.columns = ['_'.join(col).strip() for col in regional_stats.columns]

        if len(regional_stats) > 0:
            fig, axes = plt.subplots(1, 2, figsize=(15, 6))

            # Count by region
            regional_stats['market_cap_count'].plot(kind='bar', ax=axes[0], color='skyblue')
            axes[0].set_title('Stock Count by Region', fontsize=13, fontweight='bold')
            axes[0].set_xlabel('Region', fontsize=11)
            axes[0].set_ylabel('Count', fontsize=11)
            axes[0].tick_params(axis='x', rotation=45)

            # Average market cap by region
            (regional_stats['market_cap_mean'] / 1e6).plot(kind='bar', ax=axes[1], color='coral')
            axes[1].set_title('Average Market Cap by Region', fontsize=13, fontweight='bold')
            axes[1].set_xlabel('Region', fontsize=11)
            axes[1].set_ylabel('Avg Market Cap (Millions)', fontsize=11)
            axes[1].tick_params(axis='x', rotation=45)

            plt.tight_layout()
            plt.show()
            print("✓ Regional analysis charts created")

    # 4. Top Sectors Comparison Across Regions (Interactive)
    if all(['sector' in all_stocks_processed.columns,
            'region' in all_stocks_processed.columns]):

        sector_region_counts = all_stocks_processed.groupby(['sector', 'region']).size().reset_index(name='count')

        # Get top 5 sectors overall
        top_sectors = all_stocks_processed['sector'].value_counts().head(5).index
        filtered_data = sector_region_counts[sector_region_counts['sector'].isin(top_sectors)]

        if len(filtered_data) > 0:
            fig = px.bar(
                    filtered_data,
                    x='sector',
                    y='count',
                    color='region',
                    title='Top 5 Sectors Distribution Across Regions',
                    labels={'count': 'Number of Stocks', 'sector': 'Sector'},
                    barmode='group'
                    )
            fig.update_layout(height=500, xaxis_tickangle=-45)
            fig.show()
            print("✓ Interactive sector-region comparison created")

    # 5. Sector Growth Metrics Comparison
    if all(['sector' in all_stocks_processed.columns,
            'revenue_growth_1y' in all_stocks_processed.columns,
            'revenue_growth_3y' in all_stocks_processed.columns]):

        growth_data = all_stocks_processed.groupby('sector')[
            ['revenue_growth_1y', 'revenue_growth_3y']
        ].median().dropna()

        if len(growth_data) >= 5:
            # Take top sectors by count
            top_sectors_growth = all_stocks_processed['sector'].value_counts().head(10).index
            growth_data_filtered = growth_data.loc[growth_data.index.isin(top_sectors_growth)]

            if len(growth_data_filtered) > 0:
                fig, ax = plt.subplots(1, 1, figsize=(12, 8))
                x = np.arange(len(growth_data_filtered))
                width = 0.35

                ax.barh(x - width / 2, growth_data_filtered['revenue_growth_1y'], width,
                        label='1Y Growth', color='lightblue')
                ax.barh(x + width / 2, growth_data_filtered['revenue_growth_3y'], width,
                        label='3Y Growth', color='salmon')

                ax.set_yticks(x)
                ax.set_yticklabels(growth_data_filtered.index)
                ax.set_xlabel('Median Revenue Growth (%)', fontsize=11)
                ax.set_title('Revenue Growth Comparison by Sector (Top 10)',
                             fontsize=13, fontweight='bold')
                ax.legend()
                plt.tight_layout()
                plt.show()
                print("✓ Sector growth metrics comparison created")

    print("\n" + "=" * 80)
    print("✓ All sector and region analysis visualizations completed")
    print("=" * 80)

except Exception as e:
    print(f"\n✗ Error creating sector/region visualizations: {e}")
    import traceback

    traceback.print_exc()

In [ ]:
# ============================================================================
# FINANCIAL METRICS DEEP DIVE
# ============================================================================
print("\n" + "=" * 80)
print("CREATING FINANCIAL METRICS VISUALIZATIONS")
print("=" * 80)

try:
    # 1. Profitability Metrics Distribution
    profitability_metrics = ['ebitda_margin', 'net_margin', 'roe', 'roa']
    available_profit = [m for m in profitability_metrics if m in all_stocks_processed.columns]

    if len(available_profit) >= 2:
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        axes = axes.flatten()

        for idx, metric in enumerate(available_profit[:4]):
            data = all_stocks_processed[metric].dropna()
            # Filter outliers for better visualization
            data_filtered = data[(data > data.quantile(0.05)) & (data < data.quantile(0.95))]

            if len(data_filtered) > 10:
                axes[idx].hist(data_filtered, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
                axes[idx].axvline(data_filtered.mean(), color='red', linestyle='--', linewidth=2,
                                  label=f'Mean: {data_filtered.mean():.2f}')
                axes[idx].axvline(data_filtered.median(), color='green', linestyle='--', linewidth=2,
                                  label=f'Median: {data_filtered.median():.2f}')
                axes[idx].set_title(f'{metric.upper()} Distribution', fontsize=12, fontweight='bold')
                axes[idx].set_xlabel(f'{metric} (%)', fontsize=10)
                axes[idx].set_ylabel('Frequency', fontsize=10)
                axes[idx].legend()

        # Hide unused subplots
        for idx in range(len(available_profit), 4):
            axes[idx].set_visible(False)

        plt.tight_layout()
        plt.show()
        print(f"✓ Profitability metrics distributions created ({len(available_profit)} metrics)")

    # 2. Valuation Multiples Comparison
    valuation_multiples = ['p_e_ntm', 'p_b', 'p_s', 'ev_over_ebitda']
    available_multiples = [m for m in valuation_multiples if m in all_stocks_processed.columns]

    if len(available_multiples) >= 2:
        # Create violin plots for each multiple
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        axes = axes.flatten()

        for idx, metric in enumerate(available_multiples[:4]):
            data = all_stocks_processed[
                (all_stocks_processed[metric].notna()) &
                (all_stocks_processed[metric] > 0) &
                (all_stocks_processed[metric] < all_stocks_processed[metric].quantile(0.95))
                ]

            if len(data) > 20:
                parts = axes[idx].violinplot([data[metric]], positions=[0], showmeans=True, showmedians=True)
                axes[idx].set_title(f'{metric.upper()} Distribution', fontsize=12, fontweight='bold')
                axes[idx].set_ylabel(metric.upper(), fontsize=10)
                axes[idx].set_xticks([])

                # Add statistics text
                stats_text = f'Mean: {data[metric].mean():.2f}\nMedian: {data[metric].median():.2f}\nStd: {data[metric].std():.2f}'
                axes[idx].text(0.05, 0.95, stats_text,
                               transform=axes[idx].transAxes,
                               fontsize=9, verticalalignment='top',
                               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

        # Hide unused subplots
        for idx in range(len(available_multiples), 4):
            axes[idx].set_visible(False)

        plt.tight_layout()
        plt.show()
        print(f"✓ Valuation multiples visualizations created ({len(available_multiples)} multiples)")

    # 3. Leverage and Debt Metrics
    if all([m in all_stocks_processed.columns for m in ['net_debt_over_ebitda', 'debt_to_equity']]):
        fig, axes = plt.subplots(1, 2, figsize=(15, 6))

        # Net Debt/EBITDA
        debt_ebitda = all_stocks_processed['net_debt_over_ebitda'].dropna()
        debt_ebitda_filtered = debt_ebitda[(debt_ebitda > -10) & (debt_ebitda < 10)]

        if len(debt_ebitda_filtered) > 10:
            axes[0].hist(debt_ebitda_filtered, bins=30, color='salmon', edgecolor='black', alpha=0.7)
            axes[0].axvline(0, color='black', linestyle='--', linewidth=2, label='Zero Debt')
            axes[0].set_title('Net Debt / EBITDA Distribution', fontsize=13, fontweight='bold')
            axes[0].set_xlabel('Net Debt / EBITDA', fontsize=11)
            axes[0].set_ylabel('Frequency', fontsize=11)
            axes[0].legend()

        # Debt to Equity
        debt_eq = all_stocks_processed['debt_to_equity'].dropna()
        debt_eq_filtered = debt_eq[(debt_eq >= 0) & (debt_eq < debt_eq.quantile(0.95))]

        if len(debt_eq_filtered) > 10:
            axes[1].hist(debt_eq_filtered, bins=30, color='lightcoral', edgecolor='black', alpha=0.7)
            axes[1].set_title('Debt to Equity Distribution', fontsize=13, fontweight='bold')
            axes[1].set_xlabel('Debt / Equity Ratio', fontsize=11)
            axes[1].set_ylabel('Frequency', fontsize=11)

        plt.tight_layout()
        plt.show()
        print("✓ Leverage and debt metrics visualizations created")

    # 4. Growth Metrics Scatter with Profitability
    if all([m in all_stocks_processed.columns for m in ['revenue_growth_1y', 'ebitda_margin', 'market_cap']]):
        growth_profit_data = all_stocks_processed[
            (all_stocks_processed['revenue_growth_1y'].notna()) &
            (all_stocks_processed['ebitda_margin'].notna()) &
            (all_stocks_processed['market_cap'].notna()) &
            (all_stocks_processed['revenue_growth_1y'] > -50) &
            (all_stocks_processed['revenue_growth_1y'] < 100) &
            (all_stocks_processed['ebitda_margin'] > -50) &
            (all_stocks_processed['ebitda_margin'] < 100)
            ].copy()

        # Sample for performance
        if len(growth_profit_data) > 1000:
            growth_profit_data = growth_profit_data.sample(1000, random_state=42)

        if len(growth_profit_data) > 20:
            fig, ax = plt.subplots(1, 1, figsize=(12, 8))
            scatter = ax.scatter(
                    growth_profit_data['revenue_growth_1y'],
                    growth_profit_data['ebitda_margin'],
                    s=np.log10(growth_profit_data['market_cap'] + 1) * 10,
                    alpha=0.5,
                    c=growth_profit_data['market_cap'],
                    cmap='viridis',
                    edgecolors='black',
                    linewidth=0.5
                    )

            # Add quadrant lines
            ax.axhline(y=0, color='red', linestyle='--', linewidth=1, alpha=0.7)
            ax.axvline(x=0, color='red', linestyle='--', linewidth=1, alpha=0.7)

            ax.set_xlabel('Revenue Growth 1Y (%)', fontsize=12)
            ax.set_ylabel('EBITDA Margin (%)', fontsize=12)
            ax.set_title('Growth vs Profitability (Size = Market Cap)', fontsize=14, fontweight='bold')
            ax.grid(True, alpha=0.3)

            # Add colorbar
            cbar = plt.colorbar(scatter, ax=ax)
            cbar.set_label('Market Cap', fontsize=10)

            plt.tight_layout()
            plt.show()
            print("✓ Growth vs profitability scatter plot created")

    # 5. Financial Health Score (Interactive Radar Chart)
    if all([m in all_stocks_processed.columns for m in
            ['p_e_ntm', 'revenue_growth_1y', 'ebitda_margin', 'roe', 'net_debt_over_ebitda']]):
        # Calculate normalized scores (0-100 scale) for top 5 stocks by market cap
        health_metrics = ['p_e_ntm', 'revenue_growth_1y', 'ebitda_margin', 'roe', 'net_debt_over_ebitda']

        if 'market_cap' in all_stocks_processed.columns:
            top_5_stocks = all_stocks_processed.nlargest(5, 'market_cap')

            if len(top_5_stocks) > 0 and 'ticker' in top_5_stocks.columns:
                health_data = top_5_stocks[health_metrics + ['ticker']].copy()

                # Normalize each metric to 0-1 scale (handle inversions where lower is better)
                for metric in health_metrics:
                    if health_data[metric].notna().sum() > 0:
                        if metric in ['net_debt_over_ebitda', 'p_e_ntm']:  # Lower is better
                            health_data[f'{metric}_norm'] = 1 - (
                                    (health_data[metric] - health_data[metric].min()) /
                                    (health_data[metric].max() - health_data[metric].min() + 1e-6)
                            )
                        else:  # Higher is better
                            health_data[f'{metric}_norm'] = (
                                    (health_data[metric] - health_data[metric].min()) /
                                    (health_data[metric].max() - health_data[metric].min() + 1e-6)
                            )

                print("✓ Financial health metrics normalized for top 5 stocks")

    print("\n" + "=" * 80)
    print("✓ All financial metrics visualizations completed")
    print("=" * 80)

except Exception as e:
    print(f"\n✗ Error creating financial metrics visualizations: {e}")
    import traceback

    traceback.print_exc()

## Financial Metrics Deep Dive

Detailed analysis of key financial metrics and ratios.

## Sector and Region Analysis Visualizations

Advanced sector and regional comparative analysis.

## Correlation Analysis and Heatmaps

Analyze relationships between financial metrics using correlation matrices and pair plots.

## Distribution Visualizations (Matplotlib/Seaborn)

Static visualizations for data distributions and relationships.

## Enhanced Data Analytics & Visualizations

This section provides comprehensive summary statistics and visualizations for the `all_stocks` dataframe based on the implementation guide and improvement plan.